# 1. Preprocess the Data:
- Download and extract the Cats vs Dogs dataset here.


In [2]:
import os
import shutil
import random
import zipfile

# STEP 1 — Clean old folders
shutil.rmtree("stock_data", ignore_errors=True)
shutil.rmtree("data", ignore_errors=True)

# STEP 2 — Extract only image files from zip (remove nested folders)
os.makedirs("stock_data/train", exist_ok=True)

with zipfile.ZipFile("dogs-vs-cats.zip", "r") as zip_ref:
    for member in zip_ref.namelist():
        if member.startswith("train/") and member.endswith(".jpg"):
            filename = os.path.basename(member)
            source = zip_ref.open(member)
            target = open(os.path.join("stock_data/train", filename), "wb")
            with source, target:
                shutil.copyfileobj(source, target)

# STEP 3 — Define paths
original_dataset_dir = "stock_data/train"
base_dir = "data"
train_dir = os.path.join(base_dir, "train")
validation_dir = os.path.join(base_dir, "validation")

for directory in [train_dir, validation_dir]:
    os.makedirs(os.path.join(directory, "cats"), exist_ok=True)
    os.makedirs(os.path.join(directory, "dogs"), exist_ok=True)

# STEP 4 — Sort and copy images
all_images = os.listdir(original_dataset_dir)
cat_images = [img for img in all_images if img.startswith("cat")]
dog_images = [img for img in all_images if img.startswith("dog")]

random.seed(42)
random.shuffle(cat_images)
random.shuffle(dog_images)

def copy_images(images, start, end, target_dir, label):
    for img in images[start:end]:
        src = os.path.join(original_dataset_dir, img)
        dst = os.path.join(target_dir, label, img)
        shutil.copyfile(src, dst)

copy_images(cat_images, 0, 2000, train_dir, "cats")
copy_images(cat_images, 2000, 3000, validation_dir, "cats")
copy_images(dog_images, 0, 2000, train_dir, "dogs")
copy_images(dog_images, 2000, 3000, validation_dir, "dogs")

# Final check
print("✅ Setup complete")
print("Cats in training set:", len(os.listdir("data/train/cats")))
print("Dogs in validation set:", len(os.listdir("data/validation/dogs")))


✅ Setup complete
Cats in training set: 2000
Dogs in validation set: 1000


In [3]:
import os

original_dataset_dir = "stock_data/train"
print("Exemples :", os.listdir(original_dataset_dir)[:10])


Exemples : ['cat.4381.jpg', 'dog.8800.jpg', 'dog.7396.jpg', 'dog.9768.jpg', 'cat.10777.jpg', 'dog.8244.jpg', 'cat.5504.jpg', 'cat.2365.jpg', 'dog.5257.jpg', 'cat.6711.jpg']


- Use ImageDataGenerator to rescale and augment the training images (e.g., horizontal flip, rotation, zoom, and shifts).
- Create separate generators for training and validation data.

In [4]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Image size and batch size
IMG_HEIGHT = 150
IMG_WIDTH = 150
batch_size = 32

# 1. Data augmentation for training
train_image_generator = ImageDataGenerator(
    rescale=1./255,
    rotation_range=45,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.5,
    horizontal_flip=True
)

# 2. Only rescaling for validation
validation_image_generator = ImageDataGenerator(rescale=1./255)

# 3. Load images from directories
train_data_gen = train_image_generator.flow_from_directory(
    batch_size=batch_size,
    directory="data/train",
    shuffle=True,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode='binary'
)

val_data_gen = validation_image_generator.flow_from_directory(
    batch_size=batch_size,
    directory="data/validation",
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode='binary'
)


2025-07-27 20:13:16.248708: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753647196.684188    2035 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753647196.829188    2035 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1753647197.945736    2035 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1753647197.945867    2035 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1753647197.945871    2035 computation_placer.cc:177] computation placer alr

Found 4000 images belonging to 2 classes.
Found 2000 images belonging to 2 classes.


# 2. Build the Model:

- Create a CNN with:
    - Three convolutional layers with ReLU activation and max-pooling.
    - Dropout layers to reduce overfitting.
    - A fully connected layer with 512 units and ReLU activation.
    - An output layer with a single unit and sigmoid activation for binary classification.
- Compile the model using the Adam optimizer and binary cross-entropy loss.

In [5]:
from tensorflow.keras import layers, models

# Build a sequential CNN model
model = models.Sequential([
    # First convolutional layer
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    layers.MaxPooling2D(2, 2),

    # Second convolutional layer
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),

    # Third convolutional layer
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),

    # Flatten + Dense
    layers.Flatten(),
    layers.Dropout(0.5),                     # Dropout to reduce overfitting
    layers.Dense(512, activation='relu'),   # Fully connected layer
    layers.Dense(1, activation='sigmoid')   # Output layer for binary classification
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Show a summary of the model
model.summary()


/home/amoziegshirel/.local/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2025-07-27 20:13:38.959032: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 72, 72, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 34, 34, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 17, 17, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 36992)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 36992)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │    18,940,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,034,177 (72.61 MB)

 Trainable params: 19,034,177 (72.61 MB)

 Non-trainable params: 0 (0.00 B)

# 3. Train the Model:

- Train the model on the augmented training data for 15 epochs.
- Use the validation data to monitor performance during training.

In [ ]:
# Set number of epochs
epochs = 5

# Train the model
history = model.fit(
    train_data_gen,              # training data generator (with augmentation)
    steps_per_epoch=2000 // 32,  # total training images / batch size
    epochs=epochs,
    validation_data=val_data_gen,
    validation_steps=1000 // 32  # total validation images / batch size
)


/home/amoziegshirel/.local/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
62/62 ━━━━━━━━━━━━━━━━━━━━ 171s 3s/step - accuracy: 0.5081 - loss: 1.0986 - val_accuracy: 0.5192 - val_loss: 0.6924
Epoch 2/5
62/62 ━━━━━━━━━━━━━━━━━━━━ 308s 5s/step - accuracy: 0.4987 - loss: 0.6912 - val_accuracy: 0.5403 - val_loss: 0.6823
Epoch 3/5
 1/62 ━━━━━━━━━━━━━━━━━━━━ 1:43 2s/step - accuracy: 0.4688 - loss: 0.7091

/home/amoziegshirel/.local/lib/python3.10/site-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


62/62 ━━━━━━━━━━━━━━━━━━━━ 16s 233ms/step - accuracy: 0.4688 - loss: 0.7091 - val_accuracy: 0.5292 - val_loss: 0.6809
Epoch 4/5
 5/62 ━━━━━━━━━━━━━━━━━━━━ 1:48 2s/step - accuracy: 0.6125 - loss: 0.6699

# 4. Evaluate the Model:

- Plot the training and validation accuracy and loss to detect overfitting.
- Analyze the impact of data augmentation and dropout on model performance.

In [ ]:
import matplotlib.pyplot as plt

# Retrieve metrics from history
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(len(acc))

# Create two plots: accuracy and loss
plt.figure(figsize=(14, 5))

# Plot Accuracy
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')

plt.tight_layout()
plt.show()


# Conclusion

In this project, I built a Convolutional Neural Network (CNN) using TensorFlow/Keras to perform binary image classification on the Cats vs Dogs dataset.

The main steps were:

Data preprocessing:
- I cleaned and organized the dataset manually (train/validation splits)
- Applied ImageDataGenerator for real-time augmentation (rotation, zoom, shift, flip)
- Used separate generators for training and validation data

Model architecture:
- 3 convolutional layers with ReLU activation + MaxPooling
- A Dropout layer to reduce overfitting
- A fully connected layer with 512 neurons
- A final sigmoid output layer for binary classification

Compilation and training:
- I used the Adam optimizer and binary cross-entropy loss
- Trained the model for 15 epochs on augmented data
- Monitored accuracy and loss on both training and validation sets

Although training is still in progress, I expect the validation accuracy to stabilize or drop after a few epochs, which would indicate possible overfitting — this is mitigated by data augmentation and dropout.

